In [ ]:
import pandas as pd
import os
import random
import json
from dataclasses import dataclass
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# --- 1. Config Updates ---
@dataclass
class Config:
    # Update these paths to match your local folder structure
    data_dir: str = r"realwaste-main/RealWaste"
    taco_img_dir: str = r"tacoImages" 
    taco_ann_path: str = r"tacoImages/annotations.json" 
    
    image_size: int = 224
    batch_size: int = 32
    lr: float = 3e-4
    epochs: int = 30
    seed: int = 42
    num_workers: int = 2
    freeze_backbone_epochs: int = 3
    num_classes: int = 10  # Includes new 'Soft Plastics' class
    out_weights: str = "weights_efficientnet_b0_waste_10cls.pt"

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# --- 2. Domain Shift Augmentations ---
def make_transforms(image_size: int):
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.5, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
        transforms.RandomGrayscale(p=0.1), # Forces model to ignore background color bias
        transforms.RandomRotation(15),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        transforms.RandomErasing(p=0.25, value='random'),
    ])
    val_tfms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])
    return train_tfms, val_tfms

# --- 3. MergedWasteDataset Class ---
class MergedWasteDataset(Dataset):
    def __init__(self, original_ds, taco_img_dir, taco_ann_path, transform=None):
        self.original_ds = original_ds
        self.transform = transform
        
        # Mapping Logic: {TACO_ID: YOUR_CLASS_INDEX}
        # 4: Misc Trash (includes Foil), 9: Soft Plastics (includes Straws)
        self.taco_map = {
            13:0, 14:0, 15:0, 16:0,          # Cardboard
            3:1,                             # Food Organics
            6:2, 9:2, 12:2, 59:2,            # Glass
            8:3, 21:3, 22:3, 23:3, 24:3,     # Metal
            1:4, 2:4, 7:4, 17:4, 18:4, 57:4, # Misc Trash (7 is Aluminium foil)
            30:5, 31:5, 32:5, 33:5,          # Paper
            37:6, 38:6, 39:6, 42:6, 44:6,    # Plastic (Hard)
            25:7, 26:7,                      # Textile
            # Vegetation (8) is skipped - not in TACO
            34:9, 35:9, 36:9, 40:9, 41:9     # Soft Plastics (41 is Plastic straw)
        }
        
        self.taco_samples = self._load_taco(taco_img_dir, taco_ann_path)
        self.targets = self.original_ds.targets + [s[1] for s in self.taco_samples]

    def _load_taco(self, img_dir, ann_path):
        with open(ann_path, 'r') as f:
            data = json.load(f)
        
        samples = []
        for ann in data['annotations']:
            cat_id = ann['category_id']
            if cat_id in self.taco_map:
                try:
                    img_info = next(i for i in data['images'] if i['id'] == ann['image_id'])
                    path = os.path.join(img_dir, img_info['file_name'])
                    samples.append((path, self.taco_map[cat_id]))
                except StopIteration:
                    continue
        return samples

    def __len__(self):
        return len(self.original_ds) + len(self.taco_samples)

    def __getitem__(self, idx):
        # 1. Get the raw image and label based on the index
        if idx < len(self.original_ds):
            # This fetches from your original ImageFolder
            # It returns a tuple: (PIL Image, Label)
            image, label = self.original_ds[idx]
        else:
            # This fetches from the TACO file paths
            img_path, label = self.taco_samples[idx - len(self.original_ds)]
            image = Image.open(img_path).convert('RGB')
        
        # 2. CRITICAL CHANGE: Apply the transform to EVERYTHING
        # This converts the PIL Image into a PyTorch Tensor
        if self.transform:
            image = self.transform(image)
            
        return image, label

# --- 4. Model and Training Utils ---
def build_model(num_classes: int):
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def freeze_backbone(model: nn.Module, freeze: bool):
    for p in model.features.parameters():
        p.requires_grad = not freeze

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_targets.append(y.cpu().numpy())
    preds, targets = np.concatenate(all_preds), np.concatenate(all_targets)
    return accuracy_score(targets, preds), f1_score(targets, preds, average="macro")

def main():
    cfg = Config()
    set_seed(cfg.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device} | Target Classes: {cfg.num_classes}")

    train_tfms, val_tfms = make_transforms(cfg.image_size)

    # Setup original dataset source
    orig_train_base = datasets.ImageFolder(cfg.data_dir, transform=None)
    
    # Initialize the Merged Dataset
    full_ds = MergedWasteDataset(orig_train_base, cfg.taco_img_dir, cfg.taco_ann_path, transform=train_tfms)
    
    # Split using the combined targets
    indices = list(range(len(full_ds)))
    train_idx, val_idx = train_test_split(
        indices, test_size=0.2, random_state=cfg.seed, shuffle=True,
        stratify=full_ds.targets
    )

    train_ds = Subset(full_ds, train_idx)
    val_ds = Subset(full_ds, val_idx)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

    # Dynamic Weight Balancing
    train_targets = [full_ds.targets[i] for i in train_idx]
    counts = np.bincount(train_targets, minlength=cfg.num_classes)
    weights = 1.0 / np.maximum(counts, 1)
    class_weights = torch.tensor(weights / weights.sum() * cfg.num_classes, dtype=torch.float32, device=device)
    
    print("Class Counts:", counts.tolist())generate a new 
    
    model = build_model(cfg.num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

    best_f1 = -1.0
    for epoch in range(1, cfg.epochs + 1):
        # Freeze backbone for initial epochs to stabilize the new classification head
        freeze = epoch <= cfg.freeze_backbone_epochs
        freeze_backbone(model, freeze=freeze)

        # Unfreeze and lower LR for fine-tuning
        if epoch == cfg.freeze_backbone_epochs + 1:
            for g in optimizer.param_groups:
                g["lr"] = cfg.lr * 0.1
            print(f"Backbone Unfrozen. LR: {optimizer.param_groups[0]['lr']}")

        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            criterion(model(x), y).backward()
            optimizer.step()

        acc, f1 = evaluate(model, val_loader, device)
        scheduler.step(f1)
        print(f"Epoch {epoch}: Acc={acc:.4f} F1={f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), cfg.out_weights)
            print(f"Saved best weights to {cfg.out_weights}")

if __name__ == "__main__":
    main()

In [ ]:
# 1. Initialize Config and Transforms
cfg = Config()
_, val_tfms = make_transforms(cfg.image_size)

# 2. Setup the Merged Dataset (same as in main)
orig_base = datasets.ImageFolder(cfg.data_dir, transform=None)
full_ds = MergedWasteDataset(orig_base, cfg.taco_img_dir, cfg.taco_ann_path, transform=val_tfms)

# 3. Get the Indices (The Seed 42 is critical here!)
indices = list(range(len(full_ds)))
_, val_idx = train_test_split(
    indices, test_size=0.2, random_state=cfg.seed, shuffle=True,
    stratify=full_ds.targets
)

# 4. Create the Loader
val_ds = torch.utils.data.Subset(full_ds, val_idx)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers
)

print(f"Validation Loader Ready with {len(val_ds)} images.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# 1. Setup the Device and Class Names
device = "cuda" if torch.cuda.is_available() else "cpu"
class_names = [
    'Cardboard', 'Food Organics', 'Glass', 'Metal', 'Misc Trash',
    'Paper', 'Plastic (Hard)', 'Textile Trash', 'Vegetation', 'Soft Plastics'
]

def generate_confusion_matrix(model_path, val_loader):
    # 2. Rebuild the Model Architecture
    # Ensure this matches the build_model function used in training
    model = build_model(num_classes=10).to(device)
    
    # 3. Load the Saved Weights
    print(f"Loading weights from: {model_path}")
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    all_preds = []
    all_targets = []
    
    # 4. Run Inference on Validation Data
    print("Running inference...")
    with torch.no_grad():
        for images, labels in tqdm(val_loader):
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.numpy())
            
    # 5. Calculate Matrix
    cm = confusion_matrix(all_targets, all_preds)
    
    # 6. Plotting
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Waste Classification Confusion Matrix')
    plt.ylabel('Actual Label (Ground Truth)')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # 7. Print Detailed Report (Precision/Recall per class)
    print("\nDetailed Classification Report:")
    print(classification_report(all_targets, all_preds, target_names=class_names))

# --- Execution ---If Paper Recall drops: The model is being too aggressive with the "Misc" label. You can fix this by increasing the Color Jitter (to emphasize the unique colors of paper).


# Make sure your 'val_loader' is already defined from your training script
# and 'weights_efficientnet_b0_waste_10cls.pt' is in your directory.
generate_confusion_matrix("weights_efficientnet_b0_waste_10cls.pt", val_loader)